# Assignment 5: PySpark joins, window functions, Parquet and partition pruning


## 1. Imports and Spark session

The helper functions and schemas are supplied in the starter project. `get_spark()` also prints the Spark UI URL, which is useful when checking stages and SQL plans.

In [44]:
import os
import sys
from pathlib import Path

ASSIGNMENT_DIR = Path.cwd().parent
STARTER_DIR = ASSIGNMENT_DIR / "starter"
DATA_DIR = ASSIGNMENT_DIR / "data" / "olist"
sys.path.insert(0, str(STARTER_DIR))
os.environ["OLIST_DIR"] = str(DATA_DIR)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from helpers import ( # pyright: ignore[reportMissingImports]
    OUT_DIR, csv_path, get_spark, plan_text, count_files,
    folder_size_mb, fresh_dir, uri, timed
)
from pipeline import ( # pyright: ignore[reportMissingImports]
    build_sales, write_silver, compare_csv_parquet,
    latest_order_per_customer, top3_products_per_category,
    seller_monthly, read_month, add_price_band_builtin,
    add_price_band_udf, write_gold, SALES_COLUMNS
)

spark = get_spark("olist-assignment-5-notebook")

Spark 4.2.0 | Spark UI: http://Kiranmayi1814:4040


## 2. Explore the source data first

I am checking the CSV structure and the product-category mapping before joining. The category translation is not guaranteed to cover every product, so I should not use an inner join for that descriptive lookup.

In [45]:
items_raw = spark.read.csv(
    csv_path("olist_order_items_dataset.csv"),
    header=True,
    inferSchema=True
)
items_raw.printSchema()
items_raw.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee65886

In [46]:
for filename in [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "product_category_name_translation.csv",
]:
    print(f"\\n--- {filename} ---")
    spark.read.csv(csv_path(filename), header=True).show(3, truncate=35)

\n--- olist_orders_dataset.csv ---
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|                        order_id|                     customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|   delivered|     2018-07-24 20:41:37|

In [47]:
products = spark.read.csv(csv_path("olist_products_dataset.csv"), header=True)
translation = spark.read.csv(
    csv_path("product_category_name_translation.csv"), header=True
)

print("Product rows:", products.count())
print("Products with a null category:", products.filter(F.col("product_category_name").isNull()).count())

untranslated = (
    products.join(translation, "product_category_name", "left")
            .filter(
                F.col("product_category_name").isNotNull()
                & F.col("product_category_name_english").isNull()
            )
            .count()
)
print("Products whose category has no English translation:", untranslated)

Product rows: 32951
Products with a null category: 610
Products whose category has no English translation: 13


## 3. Part 1 — Build the joined sales table

The order items table is the starting point because the target grain is **one row per order item**. Every join starts from items and is a LEFT join, so an item with no match is kept and never dropped. In this data every item has an order, a customer and a product, so only the translation join would lose rows if it were an inner join (1,627 items). Missing English category labels become `unknown`.

The function reads all five CSVs with the explicit DDL schemas from `pipeline.py`.

In [48]:
sales = build_sales(spark)

print("Columns match the assignment:", sales.columns == SALES_COLUMNS)
print("Sales row count:", f"{sales.count():,}")
print("Distinct order-item keys:", f"{sales.dropDuplicates(['order_id', 'order_item_id']).count():,}")
print("Rows with null category:", sales.filter(F.col("category").isNull()).count())

sales.printSchema()
sales.show(5, truncate=False)

Columns match the assignment: True
Sales row count: 112,650
Distinct order-item keys: 112,650
Rows with null category: 0
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- purchase_ts: timestamp (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = false)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- state: string (nullable = true)

+--------------------------------+-------------+--------------------------------+-------------------+--------------------------------+--------------------------------+---------------+-----+-------------+-----+
|order_id                        |order_item_id|customer_unique_id              |purchase_ts        |seller_id                       |product_id                      |category       |price|freight_value|state|
+---------------------------

### Join check: what happens with inner joins?

This small comparison helps show why the product and translation lookups were left joins. Compare the row counts with the full `sales` result; do not assume a join is safe just because it executes.

In [49]:
items = spark.read.csv(csv_path("olist_order_items_dataset.csv"), header=True, schema=__import__("pipeline").ITEMS_DDL)
orders = spark.read.csv(csv_path("olist_orders_dataset.csv"), header=True, schema=__import__("pipeline").ORDERS_DDL)
customers = spark.read.csv(csv_path("olist_customers_dataset.csv"), header=True, schema=__import__("pipeline").CUSTOMERS_DDL)
products_typed = spark.read.csv(csv_path("olist_products_dataset.csv"), header=True, schema=__import__("pipeline").PRODUCTS_DDL)
translation_typed = spark.read.csv(csv_path("product_category_name_translation.csv"), header=True, schema=__import__("pipeline").TRANSLATION_DDL)

base_join = (
    items.join(orders.select("order_id", "customer_id"), "order_id", "inner")
         .join(customers.select("customer_id"), "customer_id", "inner")
)
with_product_left = base_join.join(products_typed.select("product_id", "product_category_name"), "product_id", "left")
with_product_inner = base_join.join(products_typed.select("product_id", "product_category_name"), "product_id", "inner")
with_translation_inner = with_product_left.join(translation_typed, "product_category_name", "inner")

print("Items:", items.count())
print("After required order/customer joins:", base_join.count())
print("After LEFT product join:", with_product_left.count())
print("After INNER product join:", with_product_inner.count())
print("After INNER translation join:", with_translation_inner.count())

Items: 112650
After required order/customer joins: 112650
After LEFT product join: 112650
After INNER product join: 112650
After INNER translation join: 111023


## 4. Part 2 — Write the Silver Parquet table

The `year_month` column is derived from `purchase_ts`. `repartition("year_month")` arranges the in-memory data by month before writing, and `partitionBy("year_month")` creates month folders on disk. The output is replaced on reruns so stale files do not remain.

In [50]:
silver_path = OUT_DIR / "silver" / "sales"
write_silver(sales, silver_path)

print("Parquet files:", count_files(silver_path))
print("Silver size (MB):", round(folder_size_mb(silver_path), 2))
print("Month folders:", len([p for p in silver_path.iterdir() if p.is_dir()]))

Parquet files: 24
Silver size (MB): 10.94
Month folders: 24


### CSV versus plain Parquet

This writes two unpartitioned comparison copies and measures their actual disk sizes. Results depend on the local Spark version and filesystem, so I record the values produced on my machine.

In [51]:
from pathlib import Path
from uuid import uuid4

# Use a fresh folder outside OneDrive
compare_workdir = Path(r"C:\spark_work") / f"compare_{uuid4().hex[:8]}"

compare_workdir.mkdir(parents=True, exist_ok=True)

# Compare CSV and Parquet
size_report = compare_csv_parquet(sales, compare_workdir)

display(size_report)

{'csv_mb': 20.924582481384277,
 'parquet_mb': 9.885944366455078,
 'csv_files': 1,
 'parquet_files': 1}

### Inspect Parquet metadata

Parquet stores column statistics in row-group metadata. The min/max values can help Spark skip row groups that cannot satisfy a filter, depending on the query and file layout.

In [ ]:
import glob
import pyarrow.parquet as pq

parquet_files = glob.glob(str(silver_path / "year_month=2018-03" / "*.parquet"))
if parquet_files:
    metadata = pq.ParquetFile(parquet_files[0]).metadata
    print("Row groups:", metadata.num_row_groups, "| Rows:", metadata.num_rows)
    for i in range(metadata.num_columns):
        column = metadata.row_group(0).column(i)
        if column.path_in_schema == "price":
            print(
                "price statistics:",
                "min =", column.statistics.min,
                "| max =", column.statistics.max,
                "| compression =", column.compression
            )
else:
    print("No March 2018 parquet file found yet. Check the Silver write step.")

## 5. Part 3 — Window functions

All three outputs are created from the Silver Parquet data, not from the original CSV joins. This makes the Silver layer the repeatable input for the reporting tables.

In [52]:
silver_df = spark.read.parquet(uri(silver_path))

latest_order = latest_order_per_customer(silver_df)
top3_products = top3_products_per_category(silver_df)
seller_monthly_df = seller_monthly(silver_df)

print("Latest order per customer:", f"{latest_order.count():,}")
print("Top products per category:", f"{top3_products.count():,}")
print("Seller monthly rows:", f"{seller_monthly_df.count():,}")

latest_order.show(5, truncate=False)
top3_products.orderBy("category", "rank").show(10, truncate=False)
seller_monthly_df.orderBy("seller_id", "year_month").show(10, truncate=False)

Latest order per customer: 95,420
Top products per category: 213
Seller monthly rows: 16,441
+--------------------------------+--------------------------------+-------------------+
|customer_unique_id              |order_id                        |purchase_ts        |
+--------------------------------+--------------------------------+-------------------+
|0000f6ccb0745a6a4b88665a16c9f078|41272756ecddd9a9ed0180413cc22fb6|2017-10-12 20:29:41|
|0004aac84e0df4da2b147fca70cf8255|d957021f1127559cd947b62533f484f7|2017-11-14 19:45:42|
|00053a61a98854899e70ed204dd4bafe|44e608f2db00c74a1fe329de44416a4e|2018-02-28 11:15:41|
|000c8bdb58a29e7115cfc257230fb21b|34801c59d6ec5c5e89bfefb6ebef314d|2017-12-12 22:53:35|
|000d460961d6dbfa3ec6c9f5805769e1|5fd641e5e16c3b056b1fb4dd6e429c43|2018-01-07 22:59:25|
+--------------------------------+--------------------------------+-------------------+
only showing top 5 rows
+--------------------------+--------------------------------+--------+----+
|category      

### Window logic notes

- `latest_order_per_customer`: `row_number()` partitions by `customer_unique_id`; sorting by timestamp descending and then `order_id` descending makes ties repeatable.
- `top3_products_per_category`: first aggregate revenue by category and product, then rank within each category. `product_id` ascending is the tie-break.
- `seller_monthly`: aggregate by seller/month, use `lag()` for the previous active month, and use an explicit unbounded-preceding-to-current-row frame for the running total.

## 6. Part 4 — Prove partition pruning

The filter is applied directly to the physical partition column `year_month`. The DataFrame plan should contain a `PartitionFilters` entry. Run an action as well so the scan appears in the Spark UI SQL tab.

In [53]:
march = read_month(spark, silver_path, "2018-03")
print("March rows:", f"{march.count():,}")
print(plan_text(march))

March rows: 8,217
*(1) ColumnarToRow
+- FileScan parquet [order_id#3065,order_item_id#3066,customer_unique_id#3067,purchase_ts#3068,seller_id#3069,product_id#3070,category#3071,price#3072,freight_value#3073,state#3074,year_month#3075] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/PAVANI/OneDrive/Desktop/OneDrive/Desktop/olist-pandas-a..., PartitionFilters: [isnotnull(year_month#3075), (year_month#3075 = 2018-03)], PushedFilters: [], ReadSchema: struct<order_id:string,order_item_id:int,customer_unique_id:string,purchase_ts:timestamp,seller_i...



### Compare with a timestamp expression

This version filters on a transformed timestamp rather than the stored partition column. Inspect the plan and compare the `PartitionFilters` entry with the previous cell.

In [54]:
march_by_timestamp = (
    spark.read.parquet(uri(silver_path))
         .filter(F.date_format("purchase_ts", "yyyy-MM") == "2018-03")
)
print("Rows from timestamp filter:", f"{march_by_timestamp.count():,}")
print(plan_text(march_by_timestamp))

Rows from timestamp filter: 8,217
*(1) Filter (isnotnull(purchase_ts#3095) AND (date_format(purchase_ts#3095, yyyy-MM, Some(Asia/Calcutta)) = 2018-03))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#3092,order_item_id#3093,customer_unique_id#3094,purchase_ts#3095,seller_id#3096,product_id#3097,category#3098,price#3099,freight_value#3100,state#3101,year_month#3102] Batched: true, DataFilters: [isnotnull(purchase_ts#3095), (date_format(purchase_ts#3095, yyyy-MM, Some(Asia/Calcutta)) = 2018..., Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/PAVANI/OneDrive/Desktop/OneDrive/Desktop/olist-pandas-a..., PartitionFilters: [], PushedFilters: [IsNotNull(purchase_ts)], ReadSchema: struct<order_id:string,order_item_id:int,customer_unique_id:string,purchase_ts:timestamp,seller_i...



## 7. Part 5 — Built-in expression versus Python UDF

Both functions should return the same labels. The built-in expression stays inside Spark's execution engine; the Python UDF introduces a Python evaluation step. Timings vary, so I run both on this machine and inspect the physical plans rather than copying a generic performance ratio.

In [55]:
built_in = add_price_band_builtin(silver_df)
python_udf = add_price_band_udf(silver_df)

print("Built-in plan:")
print(plan_text(built_in))
print("\\nUDF plan:")
print(plan_text(python_udf))

Built-in plan:
*(1) Project [order_id#2934, order_item_id#2935, customer_unique_id#2936, purchase_ts#2937, seller_id#2938, product_id#2939, category#2940, price#2941, freight_value#2942, state#2943, year_month#2944, CASE WHEN (price#2941 < 20.0) THEN budget WHEN (price#2941 < 100.0) THEN mid ELSE premium END AS price_band#3119]
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#2934,order_item_id#2935,customer_unique_id#2936,purchase_ts#2937,seller_id#2938,product_id#2939,category#2940,price#2941,freight_value#2942,state#2943,year_month#2944] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/PAVANI/OneDrive/Desktop/OneDrive/Desktop/olist-pandas-a..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_item_id:int,customer_unique_id:string,purchase_ts:timestamp,seller_i...

\nUDF plan:
*(1) Project [order_id#2934, order_item_id#2935, customer_unique_id#2936, purchase_ts#2937, seller_id#2938, product_id#

In [56]:
_, builtin_seconds = timed(
    lambda: built_in.groupBy("price_band").count().collect()
)
_, udf_seconds = timed(
    lambda: python_udf.groupBy("price_band").count().collect()
)

print(f"Built-in: {builtin_seconds:.3f} seconds")
print(f"Python UDF: {udf_seconds:.3f} seconds")
print("UDF / built-in ratio:", round(udf_seconds / builtin_seconds, 2) if builtin_seconds else None)

Built-in: 0.307 seconds
Python UDF: 21.345 seconds
UDF / built-in ratio: 69.56


## 8. Part 6 — Write the Gold tables

These reporting tables are relatively small and are expected to be read whole. `coalesce(1)` reduces each output to one file without adding an unnecessary full shuffle. Unlike Silver, Gold is not partitioned into month folders.

In [57]:
gold_path = OUT_DIR / "gold"
fresh_dir(gold_path)

gold_tables = {
    "latest_order": latest_order,
    "top3_products": top3_products,
    "seller_monthly": seller_monthly_df,
}
write_gold(gold_tables, gold_path)

for name in gold_tables:
    folder = gold_path / name
    print(name, "->", count_files(folder), "Parquet file(s)")

latest_order -> 1 Parquet file(s)
top3_products -> 1 Parquet file(s)
seller_monthly -> 1 Parquet file(s)


## 9. Ground-truth checks from the assignment

The assignment lists these expected counts for the full Olist data. Compare them with the values from the cells above. If a count differs, investigate before changing the logic.

In [58]:
expected_counts = {
    "sales": 112650,
    "latest_order": 95420,
    "top3_products": 213,
    "seller_monthly": 16441,
    "march_2018": 8217,
}
actual_counts = {
    "sales": sales.count(),
    "latest_order": latest_order.count(),
    "top3_products": top3_products.count(),
    "seller_monthly": seller_monthly_df.count(),
    "march_2018": march.count(),
}
for key, expected in expected_counts.items():
    actual = actual_counts[key]
    print(f"{key:18s} actual={actual:>8,} expected={expected:>8,} match={actual == expected}")

sales              actual= 112,650 expected= 112,650 match=True
latest_order       actual=  95,420 expected=  95,420 match=True
top3_products      actual=     213 expected=     213 match=True
seller_monthly     actual=  16,441 expected=  16,441 match=True
march_2018         actual=   8,217 expected=   8,217 match=True


### Manual seller check

The assignment asks for a direct aggregation from the raw order-items CSV and a comparison with the last-month running total for the largest seller. This cell shows the raw-source aggregation; inspect the last month for that seller in the next cell.

In [59]:
raw_items = spark.read.csv(
    csv_path("olist_order_items_dataset.csv"),
    header=True,
    inferSchema=True
)
raw_items.createOrReplaceTempView("raw_items")
spark.sql(
    """
    SELECT seller_id, COUNT(*) AS item_count, ROUND(SUM(price), 2) AS total_revenue
    FROM raw_items
    GROUP BY seller_id
    ORDER BY total_revenue DESC
    LIMIT 3
    """
).show(truncate=False)

+--------------------------------+----------+-------------+
|seller_id                       |item_count|total_revenue|
+--------------------------------+----------+-------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156      |229472.63    |
|53243585a1d6dc2643021fd1853d8905|410       |222776.05    |
|4a3ca9315b744ce9f8e9374361493884|1987      |200472.92    |
+--------------------------------+----------+-------------+



In [60]:
largest_seller = (
    spark.sql(
        """
        SELECT seller_id, ROUND(SUM(price), 2) AS total_revenue
        FROM raw_items
        GROUP BY seller_id
        ORDER BY total_revenue DESC
        LIMIT 1
        """
    ).first()
)
seller_last_month = (
    seller_monthly_df
    .filter(F.col("seller_id") == largest_seller["seller_id"])
    .orderBy(F.col("year_month").desc())
    .first()
)
print("Largest seller:", largest_seller["seller_id"])
print("Raw total revenue:", largest_seller["total_revenue"])
print("Last active month running revenue:", seller_last_month["running_revenue"])

Largest seller: 4869f7a5dfa277a7dca6462dcf3b52b2
Raw total revenue: 229472.63
Last active month running revenue: 229472.63


## 10. Experiments and reflection notes

The assignment asks for deliberate experiments and evidence from your own run. I have left these as activities instead of making up Spark UI observations or machine-specific timings.

1. Remove `partitionBy` from the latest-customer window in a scratch copy. Record the Spark warning and Stage task count, then restore it.
2. Remove `repartition("year_month")` from the Silver write in a scratch output folder. Count files per month.
3. Write a scratch copy partitioned by `seller_id`. Record folder count, file count and average file size.
4. On a **scratch copy only**, try overwriting the same Parquet path being read. Record the error and explain why transactional table formats help with this problem.
5. Compare the `PartitionFilters` lines for `year_month` and `date_format(purchase_ts, ...)`.
6. Record all three UDF/built-in timing runs, including the warm-up run.
7. Compare `row_number` and `dense_rank` for top products, and record whether ties change the number of rows.

Use the starter `REFLECTION.md` to write two to four sentences per question, with actual numbers or plan lines from your run.

## 11. Close Spark

Run this only after all the cells you need have finished.

In [61]:
spark.stop()